# Análisis espectral de la glucosa CGM — βc por grupo

Reproduce la **caracterización espectral del ruido metabólico** del Capítulo 3: estima el exponente espectral $\beta_c$ por sujeto (P(f) ∝ 1/f^βc) y compara los grupos **NDBT2** (no diabéticos) y **DBT2** (diabéticos).

**Método** (ecuaciones 3.1–3.2): se remueve la media de cada serie de glucosa y se estima la PSD por el método de Welch (`nperseg=256`). El exponente es la pendiente del ajuste log–log en banda completa, con signo cambiado.

> Detalle clave de reproducibilidad: la media se remueve a mano y se usa `detrend=False` en Welch (para no remover la media dos veces). Con eso se obtienen 2,00 / 2,57.


## 1. Configuración e imports


In [ ]:
import zipfile, glob, os, re, tempfile
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy import stats
import matplotlib.pyplot as plt

# --- Rutas a los zip (ajustar si están en otra carpeta) ---
ZIP_E = 'subject_E.zip'   # grupo DBT2 (diabéticos)
ZIP_S = 'subject_S.zip'   # grupo NDBT2 (no diabéticos)

# --- Parámetros del análisis ---
DT_MIN  = 15                 # muestreo en minutos
FS      = 60.0 / DT_MIN      # frecuencia de muestreo en ciclos/hora (= 4)
NPERSEG = 256                # longitud de segmento de Welch

# Sujetos usados (19+19): se descarta el #11 porque no existe en el grupo E
IDS = [1,2,3,4,5,6,7,8,9,10,12,13,14,15,16,17,18,19,20]

# Paleta
cE = '#d95f02'   # DBT2
cS = '#1b7837'   # NDBT2

## 2. Carga de los datos desde los zip

Cada CSV tiene una sola columna `Glucosa` con 1200 muestras (≈12,5 días).


In [ ]:
workdir = tempfile.mkdtemp()
for z in (ZIP_E, ZIP_S):
    with zipfile.ZipFile(z) as zf:
        zf.extractall(workdir)

def cargar(i, grupo):
    """Devuelve la serie de glucosa del sujeto i del grupo 'E' o 'S'."""
    ruta = os.path.join(workdir, f'subject_{i}_{grupo}.csv')
    return pd.read_csv(ruta)['Glucosa'].to_numpy(float)

print('Sujetos por grupo:', len(IDS))
print('Ejemplo (E, id 1):', cargar(1,'E')[:5], '... n =', len(cargar(1,'E')))

## 3. Estimación de βc por sujeto

PSD de Welch sobre la señal con media removida; βc = − pendiente del ajuste log–log (banda completa, f>0).


In [ ]:
def psd_welch(x):
    """PSD por Welch de la señal con su media removida."""
    f, P = welch(x - x.mean(), fs=FS, nperseg=NPERSEG, detrend=False)
    return f, P

def beta_c(f, P):
    """Exponente espectral: P(f) ~ 1/f^beta  ->  beta = -pendiente(log P vs log f)."""
    m = f > 0                      # se excluye la componente DC
    pend = np.polyfit(np.log(f[m]), np.log(P[m]), 1)[0]
    return -pend

psd_E = [psd_welch(cargar(i,'E')) for i in IDS]
psd_S = [psd_welch(cargar(i,'S')) for i in IDS]
freq  = psd_E[0][0]            # grilla de frecuencias (igual para todos)

beta_E = np.array([beta_c(f,P) for f,P in psd_E])   # DBT2
beta_S = np.array([beta_c(f,P) for f,P in psd_S])   # NDBT2

print(f'NDBT2 (S): {beta_S.mean():.2f} ± {beta_S.std(ddof=1):.2f}')
print(f'DBT2  (E): {beta_E.mean():.2f} ± {beta_E.std(ddof=1):.2f}')
print(f'Diferencia Δβc = {beta_E.mean()-beta_S.mean():.2f}')

## 4. Pruebas estadísticas

- **t de Welch** (varianzas desiguales): `ttest_ind(..., equal_var=False)`
- **Mann–Whitney U** (respaldo no paramétrico)
- **Shapiro–Wilk** (normalidad por grupo)
- **d de Cohen** (tamaño del efecto; se calcula a mano)


In [ ]:
def cohen_d(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na-1)*a.var(ddof=1) + (nb-1)*b.var(ddof=1)) / (na+nb-2))
    return (a.mean() - b.mean()) / sp

t,  p_t  = stats.ttest_ind(beta_E, beta_S, equal_var=False)   # Welch
u,  p_u  = stats.mannwhitneyu(beta_E, beta_S)                 # Mann-Whitney
_,  p_sS = stats.shapiro(beta_S)                              # normalidad NDBT2
_,  p_sE = stats.shapiro(beta_E)                              # normalidad DBT2
d        = cohen_d(beta_E, beta_S)

print(f'Welch t        : t = {t:.2f},  p = {p_t:.2e}')
print(f'Mann-Whitney   : U = {u:.0f}, p = {p_u:.2e}')
print(f'Shapiro NDBT2  : p = {p_sS:.3f}')
print(f'Shapiro DBT2   : p = {p_sE:.3f}')
print(f'Cohen d        : {d:.2f}')

## 5. Figura — espectro PSD medio por grupo


In [ ]:
mPSD_E = np.mean([P for _,P in psd_E], axis=0)
mPSD_S = np.mean([P for _,P in psd_S], axis=0)
m = freq > 0

fig, ax = plt.subplots(figsize=(7,4.5))
ax.loglog(freq[m], mPSD_S[m], color=cS, lw=2, label=f'NDBT2   βc = {beta_S.mean():.2f}')
ax.loglog(freq[m], mPSD_E[m], color=cE, lw=2, label=f'DBT2    βc = {beta_E.mean():.2f}')
for mu, c in [(mPSD_S, cS), (mPSD_E, cE)]:
    cf = np.polyfit(np.log(freq[m]), np.log(mu[m]), 1)
    ax.loglog(freq[m], np.exp(np.polyval(cf, np.log(freq[m]))), '--', color=c, lw=1, alpha=.7)
ax.set_xlabel('Frecuencia (ciclos/hora)')
ax.set_ylabel('PSD (Welch)')
ax.legend(); ax.grid(True, which='both', alpha=.25)
fig.tight_layout()
fig.savefig('fig_espectro_PSD.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Figura — distribución de βc por grupo


In [ ]:
fig, ax = plt.subplots(figsize=(5.5,4.5))
datos, pos, cols = [beta_S, beta_E], [1,2], [cS, cE]
bp = ax.boxplot(datos, positions=pos, widths=.5, patch_artist=True, showfliers=False)
for caja, c in zip(bp['boxes'], cols): caja.set_facecolor(c); caja.set_alpha(.35)
for med in bp['medians']: med.set_color('k')
rng = np.random.default_rng(0)
for dd, p, c in zip(datos, pos, cols):
    ax.scatter(p + rng.uniform(-.12,.12,len(dd)), dd, color=c, s=28, edgecolor='k', lw=.4, zorder=3)
ax.set_xticks(pos); ax.set_xticklabels(['NDBT2','DBT2'])
ax.set_ylabel('Exponente espectral βc')
ax.grid(True, axis='y', alpha=.25)
fig.tight_layout()
fig.savefig('fig_beta_por_grupo.png', dpi=200, bbox_inches='tight')
plt.show()

## Notas

- **βc absoluto vs. diferencia.** El valor absoluto de βc depende del preprocesamiento; el descriptor robusto es la **diferencia Δβc ≈ 0,57** entre grupos, invariante frente a esa elección.
- **Por qué Welch y no Student.** Las varianzas son muy distintas (0,09 vs 0,27); `equal_var=False` evita asumir igualdad de varianzas.
- **Escala de frecuencia.** `FS` solo fija el eje x; la pendiente (y por lo tanto βc) es invariante al valor de `FS`.
